# 05 - Recommendations
## NYC Taxi Revenue Optimization Project

**Purpose:**
Build aggregated summary tables optimized for Power BI dashboard.
Each table directly answers one dimension of the business question.

**Output Tables:**
1. `workspace.nyc_taxi.rec_revenue_by_hour`
2. `workspace.nyc_taxi.rec_revenue_by_zone`
3. `workspace.nyc_taxi.rec_revenue_by_route`
4. `workspace.nyc_taxi.rec_revenue_by_day`
5. `workspace.nyc_taxi.rec_zone_hour_matrix`

**Source:** `workspace.nyc_taxi.yellow_trips_cleaned`
**Filter:** 2023–2025 only — post-fare-restructure baseline

### Step 1 — Revenue by Hour Table

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.nyc_taxi.rec_revenue_by_hour AS
SELECT
    pickup_hour                             AS hour,
    COUNT(*)                                AS total_trips,
    ROUND(AVG(total_amount), 2)             AS avg_rev_per_trip,
    ROUND(AVG(revenue_per_mile), 2)         AS avg_rev_per_mile,
    ROUND(AVG(revenue_per_minute), 2)       AS avg_rev_per_min,
    ROUND(AVG(tip_pct), 2)                  AS avg_tip_pct,
    ROUND(AVG(trip_minutes), 2)             AS avg_trip_mins
FROM workspace.nyc_taxi.yellow_trips_cleaned
WHERE data_year IN (2023, 2024, 2025)
GROUP BY pickup_hour
ORDER BY pickup_hour;

SELECT * FROM workspace.nyc_taxi.rec_revenue_by_hour;

### Step 2 — Revenue by Zone Table

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.nyc_taxi.rec_revenue_by_zone AS
SELECT
    PULocationID                                AS zone_id,
    COUNT(*)                                    AS total_trips,
    ROUND(AVG(total_amount), 2)                 AS avg_rev_per_trip,
    ROUND(AVG(revenue_per_mile), 2)             AS avg_rev_per_mile,
    ROUND(AVG(revenue_per_minute), 2)           AS avg_rev_per_min,
    ROUND(AVG(tip_pct), 2)                      AS avg_tip_pct,
    ROUND(AVG(trip_minutes), 2)                 AS avg_trip_mins,
    CASE WHEN PULocationID IN (1, 132, 138)
         THEN 'Airport' ELSE 'City'
    END                                         AS zone_type
FROM workspace.nyc_taxi.yellow_trips_cleaned
WHERE data_year IN (2023, 2024, 2025)
GROUP BY PULocationID
HAVING COUNT(*) >= 10000
ORDER BY avg_rev_per_trip DESC;

SELECT * FROM workspace.nyc_taxi.rec_revenue_by_zone;

### Step 3 — Revenue by Route Type Table

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.nyc_taxi.rec_revenue_by_route AS
SELECT
    CASE
        WHEN PULocationID IN (132, 138, 1)
            THEN 'Airport Pickup'
        WHEN DOLocationID IN (132, 138, 1)
         AND PULocationID NOT IN (132, 138, 1)
            THEN 'City to Airport'
        ELSE 'City to City'
    END                                         AS route_type,
    COUNT(*)                                    AS total_trips,
    ROUND(AVG(total_amount), 2)                 AS avg_rev_per_trip,
    ROUND(AVG(revenue_per_mile), 2)             AS avg_rev_per_mile,
    ROUND(AVG(revenue_per_minute), 2)           AS avg_rev_per_min,
    ROUND(AVG(tip_pct), 2)                      AS avg_tip_pct,
    ROUND(AVG(trip_minutes), 2)                 AS avg_trip_mins,
    ROUND(AVG(trip_distance), 2)                AS avg_trip_miles
FROM workspace.nyc_taxi.yellow_trips_cleaned
WHERE data_year IN (2023, 2024, 2025)
GROUP BY route_type
ORDER BY avg_rev_per_trip DESC;

SELECT * FROM workspace.nyc_taxi.rec_revenue_by_route;

### Step 4 — Revenue by Day of Week Table

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.nyc_taxi.rec_revenue_by_day AS
SELECT
    DAYOFWEEK(tpep_pickup_datetime)             AS day_number,
    DATE_FORMAT(tpep_pickup_datetime, 'EEEE')   AS day_name,
    COUNT(*)                                    AS total_trips,
    ROUND(AVG(total_amount), 2)                 AS avg_rev_per_trip,
    ROUND(AVG(revenue_per_mile), 2)             AS avg_rev_per_mile,
    ROUND(AVG(revenue_per_minute), 2)           AS avg_rev_per_min,
    ROUND(AVG(tip_pct), 2)                      AS avg_tip_pct,
    ROUND(AVG(trip_minutes), 2)                 AS avg_trip_mins
FROM workspace.nyc_taxi.yellow_trips_cleaned
WHERE data_year IN (2023, 2024, 2025)
GROUP BY day_number, day_name
ORDER BY day_number;

SELECT * FROM workspace.nyc_taxi.rec_revenue_by_day;

### Step 5 — Zone Hour Matrix Table

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.nyc_taxi.rec_zone_hour_matrix AS
SELECT
    PULocationID                                AS zone_id,
    CASE WHEN PULocationID IN (1, 132, 138)
         THEN 'Airport' ELSE 'City'
    END                                         AS zone_type,
    pickup_hour                                 AS hour,
    COUNT(*)                                    AS total_trips,
    ROUND(AVG(total_amount), 2)                 AS avg_rev_per_trip,
    ROUND(AVG(revenue_per_minute), 2)           AS avg_rev_per_min,
    ROUND(AVG(tip_pct), 2)                      AS avg_tip_pct,
    ROUND(AVG(trip_minutes), 2)                 AS avg_trip_mins
FROM workspace.nyc_taxi.yellow_trips_cleaned
WHERE data_year IN (2023, 2024, 2025)
GROUP BY PULocationID, zone_type, pickup_hour
HAVING COUNT(*) >= 5000
ORDER BY avg_rev_per_trip DESC;

SELECT * FROM workspace.nyc_taxi.rec_zone_hour_matrix;

In [0]:
%sql
